<a href="https://colab.research.google.com/github/jingwen320/skinmate/blob/feature%2Fmodel/skinmate_model2_enhanced.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
import os

# 1. Mount the Drive
drive.mount('/content/drive', force_remount=True)

# 2. Verify the file exists before unzipping
zip_path = '/content/drive/MyDrive/skinmate_dataset/skin_conditions.zip'

if os.path.exists(zip_path):
    print("✅ Success! skin_conditions.zip found.")
else:
    print("❌ Still can't find it. Current files in that folder:")
    # This helps us see if there is a typo in the folder name
    if os.path.exists('/content/drive/MyDrive/skinmate_dataset'):
        print(os.listdir('/content/drive/MyDrive/skinmate_dataset'))
    else:
        print("The folder 'skinmate_dataset' doesn't seem to exist in MyDrive.")

Mounted at /content/drive
✅ Success! skin_conditions.zip found.


In [ ]:
import zipfile
import os
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau

# 1. Prepare Data
extract_path = '/content/skinmate_conditions_data_enhanced'

if not os.path.exists(extract_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("✅ Dataset unzipped!")

# 2. Independent Data Generators
IMG_SIZE = (300, 300)
BATCH_SIZE = 32

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=15,
    horizontal_flip=True
)

train_gen = datagen.flow_from_directory(
    extract_path, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='training'
)

val_gen = datagen.flow_from_directory(
    extract_path, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='validation'
)

# 3. Build "Fairness" Model
base_model = tf.keras.applications.EfficientNetB0(weights='imagenet', include_top=False, input_shape=(300, 300, 3))
base_model.trainable = True

x = tf.keras.layers.GlobalAveragePooling2D()(base_model.output)
x = tf.keras.layers.Dropout(0.4)(x)
# 6 Units + Sigmoid = Independent Marks
predictions = tf.keras.layers.Dense(6, activation='sigmoid')(x)

model2 = tf.keras.models.Model(inputs=base_model.input, outputs=predictions)

model2.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# 4. Progress Saving (Model 2)
checkpoint = ModelCheckpoint(
    filepath='/content/drive/MyDrive/skinmate_conditions_best_v2.keras',
    monitor='val_loss',
    save_best_only=True,
    mode='min',
    verbose=1
)

reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)

class_weights = {
    0: 1.0,
    1: 1.0,
    2: 1.0,
    3: 1.0,
    4: 2.0,
    5: 1.0
}

# 5. Start Training
print("🚀 Training Model 2: Independent Skin Conditions...")
model2.fit(
    train_gen,
    validation_data=val_gen,
    epochs=50,
    callbacks=[checkpoint, reduce_lr],
    class_weight=class_weights
)

Found 2695 images belonging to 6 classes.
Found 672 images belonging to 6 classes.
🚀 Training Model 2: Independent Skin Conditions...
Epoch 1/50
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 21s/step - accuracy: 0.5037 - loss: 0.5426 
Epoch 1: val_loss improved from None to 0.76942, saving model to /content/drive/MyDrive/skinmate_conditions_best_v2.keras

Epoch 1: finished saving model to /content/drive/MyDrive/skinmate_conditions_best_v2.keras
85/85 ━━━━━━━━━━━━━━━━━━━━ 1992s 23s/step - accuracy: 0.6523 - loss: 0.3995 - val_accuracy: 0.1786 - val_loss: 0.7694 - learning_rate: 1.0000e-04
Epoch 2/50
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 21s/step - accuracy: 0.8417 - loss: 0.1798 
Epoch 2: val_loss improved from 0.76942 to 0.67590, saving model to /content/drive/MyDrive/skinmate_conditions_best_v2.keras

Epoch 2: finished saving model to /content/drive/MyDrive/skinmate_conditions_best_v2.keras
85/85 ━━━━━━━━━━━━━━━━━━━━ 1952s 22s/step - accuracy: 0.8568 - loss: 0.1570 - val_accuracy: 0.1845 - val_loss: 0.6759 -

In [ ]:
import os
from PIL import Image

data_path = '/content/skinmate_conditions_data_enhanced'
bad_files = 0

print("🔍 Scanning for corrupted images...")

for root, dirs, files in os.walk(data_path):
    for file in files:
        file_path = os.path.join(root, file)
        try:
            # Try to open the image with PIL
            with Image.open(file_path) as img:
                img.verify() # Verify it's not corrupted
        except Exception:
            print(f"❌ Removing corrupted/invalid file: {file_path}")
            os.remove(file_path)
            bad_files += 1

print(f"\n✅ Cleanup finished. Removed {bad_files} bad files.")

🔍 Scanning for corrupted images...
❌ Removing corrupted/invalid file: /content/skinmate_conditions_data_enhanced/redness/.DS_Store
❌ Removing corrupted/invalid file: /content/skinmate_conditions_data_enhanced/acne/.DS_Store
❌ Removing corrupted/invalid file: /content/skinmate_conditions_data_enhanced/pores/.DS_Store
❌ Removing corrupted/invalid file: /content/skinmate_conditions_data_enhanced/wrinkles/.DS_Store
❌ Removing corrupted/invalid file: /content/skinmate_conditions_data_enhanced/__MACOSX/._wrinkles
❌ Removing corrupted/invalid file: /content/skinmate_conditions_data_enhanced/__MACOSX/._acne
❌ Removing corrupted/invalid file: /content/skinmate_conditions_data_enhanced/__MACOSX/._dark_spots
❌ Removing corrupted/invalid file: /content/skinmate_conditions_data_enhanced/__MACOSX/._pigmentation
❌ Removing corrupted/invalid file: /content/skinmate_conditions_data_enhanced/__MACOSX/._redness
❌ Removing corrupted/invalid file: /content/skinmate_conditions_data_enhanced/__MACOSX/._pores

In [ ]:
import shutil
import os

# Path to the unzipped folder
macosx_path = '/content/skinmate_conditions_data_enhanced/__MACOSX'

if os.path.exists(macosx_path):
    shutil.rmtree(macosx_path)
    print("🧹 Successfully deleted hidden __MACOSX folder.")
else:
    print("No __MACOSX folder found. Check for other hidden folders in your class_indices.")

🧹 Successfully deleted hidden __MACOSX folder.


In [ ]:
print(train_gen.class_indices)

{'__MACOSX': 0, 'acne': 1, 'dark_spots': 2, 'pigmentation': 3, 'pores': 4, 'redness': 5, 'wrinkles': 6}
